# Práctica 04 — Representaciones dispersas

**Curso:** NLP Clásico con Python · **Clase 04** · **Tiempo estimado:** 60-75 minutos
**Notebook de clase:** `notebooks_clase/clase04_representaciones_dispersas.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Los textos de los ejercicios 1 a 5 están en la celda de datos que sigue al setup; el reto usa las reseñas de `datos/` (en Colab, sube la carpeta `datos/`). Son textos sintéticos escritos para el curso (*ejemplo ilustrativo*).
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. Librerías: `numpy`, `pandas`, `scipy` y `scikit-learn`.

---
## Celda 0 — Setup y datos

In [ ]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

ARCHIVO_RESENAS = {"spa": "resenas_es.csv", "ita": "resenas_it.csv"}
print("Librerías cargadas correctamente.")

In [ ]:
# datos-es
FRASES = {
    "spa": ["la entrega fue rápida",
           "la entrega fue lenta",
           "el servicio fue rápido y amable",
           "la atención fue amable"],
    "ita": ["la consegna è stata veloce",
           "la consegna è stata lenta",
           "il servizio è stato veloce e cortese",
           "il personale è stato cortese"],
}
FRASE_NEGACION = {"spa": "no es bueno, no es malo", "ita": "non è buono, non è cattivo"}

## Ejercicio 1 — Bolsa de palabras a mano

Completa `bolsa(frases)`: debe devolver `(vocabulario, matriz)`, donde `vocabulario` es la lista **ordenada** de palabras distintas (tokenizadas con `\w+` y en minúsculas) y `matriz[i][j]` es el número de veces que `vocabulario[j]` aparece en la frase `i` (lista de listas de enteros). Se comprueba con `CountVectorizer` en los dos idiomas.

Antes de programar, cuenta a mano cuántas palabras distintas tiene `FRASES["spa"]`.

In [ ]:
def bolsa(frases):
    tokens = [re.findall(r"\w+", f.lower()) for f in frases]
    vocabulario = None      # TODO: lista ordenada de palabras distintas
    matriz = None           # TODO: lista de listas de recuentos
    return vocabulario, matriz

vocab_es, matriz_es = bolsa(FRASES["spa"])
vocab_it, matriz_it = bolsa(FRASES["ita"])

# Comprobación automática
assert vocab_es is not None and matriz_es is not None and vocab_it is not None and matriz_it is not None, "Completa las líneas marcadas con TODO"
assert len(vocab_es) == 11 and len(vocab_it) == 12, "el vocabulario tiene 11 palabras en español y 12 en italiano"
assert [sum(f) for f in matriz_es] == [4, 4, 6, 4], "la suma de cada fila es el número de palabras de la frase"
assert [sum(f) for f in matriz_it] == [5, 5, 7, 5], "la suma de cada fila es el número de palabras de la frase"
for idioma, vocab, matriz in (("spa", vocab_es, matriz_es), ("ita", vocab_it, matriz_it)):
    cv = CountVectorizer(token_pattern=r"\w+")
    esperada = cv.fit_transform(FRASES[idioma]).toarray()
    assert vocab == list(cv.get_feature_names_out()), "el vocabulario debe estar ordenado y sin repeticiones"
    assert (np.array(matriz) == esperada).all(), "la matriz no coincide con la de CountVectorizer"
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — idf y tf-idf a mano

Sobre `FRASES["spa"]` (`n = 4` frases) y con la fórmula de scikit-learn `idf = ln((1 + n) / (1 + df)) + 1`:

- completa la función `idf(n, df)`,
- calcula `idf_entrega` (la palabra `entrega`),
- calcula el tf-idf **sin normalizar** de cada palabra de la frase 0 (`pesos_brutos`: diccionario palabra → `tf · idf`),
- calcula `peso_entrega`: el peso de `entrega` tras dividir entre la norma L2 de la fila.

Comprueba antes a mano cuántas frases contienen `entrega`.

In [ ]:
def idf(n, df):
    return None      # TODO: fórmula de scikit-learn

tokens = [re.findall(r"\w+", f.lower()) for f in FRASES["spa"]]
n = len(tokens)
df_palabra = Counter(p for fila in tokens for p in set(fila))       # frases que contienen cada palabra

idf_entrega = None       # TODO
pesos_brutos = None      # TODO: {palabra: tf · idf} para la frase 0
peso_entrega = None      # TODO: pesos_brutos["entrega"] / norma L2 de la fila

# Comprobación automática
assert idf_entrega is not None and pesos_brutos is not None and peso_entrega is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(idf(4, 4), 1.0), "una palabra presente en todas las frases tiene idf = 1"
tv = TfidfVectorizer(token_pattern=r"\w+").fit(FRASES["spa"])
assert np.isclose(idf_entrega, tv.idf_[tv.vocabulary_["entrega"]]), "revisa la fórmula: ln((1 + n) / (1 + df)) + 1"
assert np.isclose(idf_entrega, np.log(5 / 3) + 1), "entrega aparece en 2 de las 4 frases: ln(5/3) + 1"
bruta = TfidfVectorizer(token_pattern=r"\w+", norm=None).fit(FRASES["spa"])
fila0 = bruta.transform(FRASES["spa"][:1]).toarray()[0]
assert set(pesos_brutos) == set(tokens[0]), "pesos_brutos debe tener una entrada por cada palabra distinta de la frase 0"
for palabra, peso in pesos_brutos.items():
    assert np.isclose(peso, fila0[bruta.vocabulary_[palabra]]), f"peso incorrecto para {palabra}"
assert np.isclose(peso_entrega, tv.transform(FRASES["spa"][:1])[0, tv.vocabulary_["entrega"]]), "el peso normalizado es el valor bruto dividido entre la norma L2 de la fila"
print(f"Ejercicio 2 correcto ✔ (idf de entrega = {idf_entrega:.4f}, peso normalizado = {peso_entrega:.4f})")

## Ejercicio 3 — Contar bigramas

`FRASE_NEGACION` tiene una frase por idioma con un bigrama repetido. Completa `bigramas(frase)` (lista de parejas de palabras consecutivas, con `\w+` y en minúsculas) y calcula:

- `n_total`: cuántos bigramas hay (con repeticiones) en la frase en español,
- `n_distintos`: cuántos son distintos,
- `n_rasgos_1_2`: cuántas características tendría un vectorizador con `ngram_range=(1, 2)` entrenado solo con esa frase (unigramas distintos + bigramas distintos).

Escribe primero los bigramas de la frase en el papel.

In [ ]:
def bigramas(frase):
    tokens = re.findall(r"\w+", frase.lower())
    return None      # TODO: lista de tuplas (palabra_i, palabra_i+1)

n_total = None           # TODO
n_distintos = None       # TODO
n_rasgos_1_2 = None      # TODO

# Comprobación automática
assert n_total is not None and n_distintos is not None and n_rasgos_1_2 is not None and bigramas("a b") is not None, "Completa las líneas marcadas con TODO"
assert bigramas("uno dos tres") == [("uno", "dos"), ("dos", "tres")], "un bigrama es una pareja de palabras consecutivas"
assert n_total == 5 and n_distintos == 4, "la frase tiene 5 bigramas y el primero se repite"
assert n_rasgos_1_2 == 8, "4 palabras distintas + 4 bigramas distintos"
for idioma, frase in FRASE_NEGACION.items():
    cv_bi = CountVectorizer(token_pattern=r"\w+", ngram_range=(2, 2)).fit([frase])
    assert len(cv_bi.vocabulary_) == len(set(bigramas(frase))) == 4
    cv_12 = CountVectorizer(token_pattern=r"\w+", ngram_range=(1, 2)).fit([frase])
    assert len(cv_12.vocabulary_) == 8
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Similitud del coseno a mano

Tienes tres vectores de recuentos `u`, `v` y `w` (con `w = 2u`). Implementa `coseno(a, b)` con `numpy` y calcula `cos_uv`, `cos_uw` y `cos_vw`. Calcula antes a mano `cos(u, v)` = `(u · v) / (‖u‖ ‖v‖)` con `u = [2, 1, 0, 1]` y `v = [1, 1, 1, 0]`.

Pregunta: ¿por qué `cos_uw` vale exactamente 1 aunque `w` tenga el doble de recuentos?

In [ ]:
u = np.array([2, 1, 0, 1])
v = np.array([1, 1, 1, 0])
w = 2 * u

def coseno(a, b):
    return None      # TODO: producto escalar entre el producto de las normas

cos_uv = None        # TODO
cos_uw = None        # TODO
cos_vw = None        # TODO

# Comprobación automática
assert cos_uv is not None and cos_uw is not None and cos_vw is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(cos_uv, 3 / np.sqrt(18)), "u·v = 3, ‖u‖ = √6, ‖v‖ = √3"
assert np.isclose(cos_uw, 1.0), "w tiene la misma dirección que u"
assert np.isclose(cos_vw, cos_uv), "v se compara con la misma dirección, así que cos(v, w) = cos(v, u)"
esperada = cosine_similarity(np.array([u, v, w]))
assert np.allclose([cos_uv, cos_uw, cos_vw], [esperada[0, 1], esperada[0, 2], esperada[1, 2]]), "debe coincidir con cosine_similarity de scikit-learn"
assert cos_uw > cos_uv, "u está más cerca de w que de v"
print(f"Ejercicio 4 correcto ✔ (cos(u, v) = {cos_uv:.4f})")

## Ejercicio 5 — Predecir el tamaño del vocabulario con `min_df` y `max_df`

Con las cuatro frases de cada idioma (`n = 4`) calcula **sin usar `CountVectorizer`** cuántas palabras tendría el vocabulario con:

- `base`: sin restricciones,
- `min2`: `min_df=2` (palabras en al menos 2 frases),
- `max075`: `max_df=0.75` (palabras en como mucho el 75 % de las frases, es decir, en 3 de 4),
- `ambos`: las dos condiciones a la vez.

Completa `df_por_palabra` y `predecir`, que devuelve la tupla `(base, min2, max075, ambos)`. Después se compara con `CountVectorizer`.

In [ ]:
def df_por_palabra(frases):
    return None      # TODO: Counter palabra → número de frases que la contienen

def predecir(frases):
    n = len(frases)
    df = df_por_palabra(frases)
    base = None      # TODO
    min2 = None      # TODO
    max075 = None    # TODO
    ambos = None     # TODO
    return base, min2, max075, ambos

previsto = {idioma: predecir(FRASES[idioma]) for idioma in ("spa", "ita")}

# Comprobación automática
assert df_por_palabra(["a b", "a c"]) is not None and all(v is not None for t in previsto.values() for v in t), "Completa las líneas marcadas con TODO"
assert previsto["spa"] == (11, 4, 10, 3), "español: 11 palabras; con min_df=2 quedan la, fue, entrega y amable; con max_df=0.75 se va fue (está en las 4 frases)"
assert previsto["ita"] == (12, 8, 11, 7), "italiano: 12 palabras; con max_df=0.75 se va è (está en las 4 frases)"
for idioma in ("spa", "ita"):
    real = tuple(len(CountVectorizer(token_pattern=r"\w+", **p).fit(FRASES[idioma]).vocabulary_)
                 for p in ({}, {"min_df": 2}, {"max_df": 0.75}, {"min_df": 2, "max_df": 0.75}))
    assert real == previsto[idioma], f"{idioma}: tu previsión {previsto[idioma]} no coincide con CountVectorizer {real}"
print("Ejercicio 5 correcto ✔")

---
## Reto final — Un buscador de «reseña más parecida»

Construye un pequeño motor de similitud sobre las **reseñas limpias** (`texto_limpio`) de `datos/`. Elige el idioma con `IDIOMA` (`"spa"` o `"ita"`).

1. Crea `vectorizador` (un `TfidfVectorizer` con `token_pattern=r"\w+"`) y ajústalo con `df_res["texto_limpio"]`; guarda la matriz en `X` (debe ser una matriz **dispersa**).
2. Escribe `mas_similares(i, k=3)`: los índices de las `k` reseñas más parecidas a la reseña `i` (por coseno), **sin incluir la propia `i`**, de mayor a menor similitud.
3. Escribe `buscar(consulta, k=3)`: transforma un texto nuevo con el vectorizador ya ajustado (`transform`, no `fit`) y devuelve los índices de las `k` reseñas más parecidas.

> **Nota:** con 300 reseñas sintéticas el motor encuentra reseñas con **palabras** parecidas, no necesariamente con el mismo significado, y las cifras no se generalizan a colecciones grandes. Sirve para practicar la lógica de la representación dispersa.

In [ ]:
IDIOMA = "spa"            # o "ita"
df_res = pd.read_csv(cargar_datos(ARCHIVO_RESENAS[IDIOMA]))

vectorizador = None      # TODO: TfidfVectorizer(token_pattern=r"\w+")
X = None                 # TODO: ajusta y transforma df_res["texto_limpio"]

def mas_similares(i, k=3):
    # TODO: coseno de la fila i con todas las filas; excluye i; ordena de mayor a menor
    return None

def buscar(consulta, k=3):
    # TODO: transforma la consulta con el vectorizador ya ajustado
    return None

# Comprobación automática
assert vectorizador is not None and X is not None, "Completa las líneas marcadas con TODO"
assert sp.issparse(X) and X.shape[0] == len(df_res), "X debe ser una matriz dispersa con una fila por reseña"
assert X.nnz / (X.shape[0] * X.shape[1]) < 0.05, "la matriz debería tener menos del 5 % de celdas no nulas"
assert np.allclose(np.sqrt(X.multiply(X).sum(axis=1)).A1, 1.0), "las filas de TfidfVectorizer tienen norma L2 igual a 1"
for i in range(0, len(df_res), 25):
    vecinos = mas_similares(i, k=3)
    assert vecinos is not None and len(vecinos) == 3 and i not in list(vecinos), "mas_similares debe devolver 3 índices sin incluir la propia reseña"
    sim = cosine_similarity(X[i], X[list(vecinos)]).ravel()
    assert (np.diff(sim) <= 1e-12).all(), "los vecinos deben ir de mayor a menor similitud"
    assert buscar(df_res.loc[i, "texto_limpio"], k=1)[0] == i, "una reseña debe ser su propia vecina más parecida al buscarla"
consulta_nueva = df_res.loc[3, "texto_limpio"].lower()
assert buscar(consulta_nueva, k=1)[0] == 3, "el texto en minúsculas sigue encontrando la reseña original"
print("Reto final correcto ✔")

**Tus conclusiones (escribe 3-4 frases):** en el reto, ¿los vecinos de una reseña tienen el mismo sentimiento? Mira dos o tres ejemplos con `df_res["etiqueta"]`. ¿Qué cambiarías (n-gramas, `min_df`, `max_df`, caracteres) para mejorar el buscador y cómo medirías si mejora?

*(Tu respuesta aquí.)*

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
def bolsa(frases):
    tokens = [re.findall(r"\w+", f.lower()) for f in frases]
    vocabulario = sorted({t for fila in tokens for t in fila})
    matriz = [[fila.count(palabra) for palabra in vocabulario] for fila in tokens]
    return vocabulario, matriz

vocab_es, matriz_es = bolsa(FRASES["spa"])
vocab_it, matriz_it = bolsa(FRASES["ita"])

# Comprobación automática
assert vocab_es is not None and matriz_es is not None and vocab_it is not None and matriz_it is not None, "Completa las líneas marcadas con TODO"
assert len(vocab_es) == 11 and len(vocab_it) == 12, "el vocabulario tiene 11 palabras en español y 12 en italiano"
assert [sum(f) for f in matriz_es] == [4, 4, 6, 4], "la suma de cada fila es el número de palabras de la frase"
assert [sum(f) for f in matriz_it] == [5, 5, 7, 5], "la suma de cada fila es el número de palabras de la frase"
for idioma, vocab, matriz in (("spa", vocab_es, matriz_es), ("ita", vocab_it, matriz_it)):
    cv = CountVectorizer(token_pattern=r"\w+")
    esperada = cv.fit_transform(FRASES[idioma]).toarray()
    assert vocab == list(cv.get_feature_names_out()), "el vocabulario debe estar ordenado y sin repeticiones"
    assert (np.array(matriz) == esperada).all(), "la matriz no coincide con la de CountVectorizer"
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
def idf(n, df):
    return float(np.log((1 + n) / (1 + df)) + 1)

tokens = [re.findall(r"\w+", f.lower()) for f in FRASES["spa"]]
n = len(tokens)
df_palabra = Counter(p for fila in tokens for p in set(fila))

idf_entrega = idf(n, df_palabra["entrega"])
pesos_brutos = {p: veces * idf(n, df_palabra[p]) for p, veces in Counter(tokens[0]).items()}
peso_entrega = pesos_brutos["entrega"] / np.sqrt(sum(v ** 2 for v in pesos_brutos.values()))

# Comprobación automática
assert idf_entrega is not None and pesos_brutos is not None and peso_entrega is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(idf(4, 4), 1.0), "una palabra presente en todas las frases tiene idf = 1"
tv = TfidfVectorizer(token_pattern=r"\w+").fit(FRASES["spa"])
assert np.isclose(idf_entrega, tv.idf_[tv.vocabulary_["entrega"]]), "revisa la fórmula: ln((1 + n) / (1 + df)) + 1"
assert np.isclose(idf_entrega, np.log(5 / 3) + 1), "entrega aparece en 2 de las 4 frases: ln(5/3) + 1"
bruta = TfidfVectorizer(token_pattern=r"\w+", norm=None).fit(FRASES["spa"])
fila0 = bruta.transform(FRASES["spa"][:1]).toarray()[0]
assert set(pesos_brutos) == set(tokens[0]), "pesos_brutos debe tener una entrada por cada palabra distinta de la frase 0"
for palabra, peso in pesos_brutos.items():
    assert np.isclose(peso, fila0[bruta.vocabulary_[palabra]]), f"peso incorrecto para {palabra}"
assert np.isclose(peso_entrega, tv.transform(FRASES["spa"][:1])[0, tv.vocabulary_["entrega"]]), "el peso normalizado es el valor bruto dividido entre la norma L2 de la fila"
print(f"Ejercicio 2 correcto ✔ (idf de entrega = {idf_entrega:.4f}, peso normalizado = {peso_entrega:.4f})")

### Solución del ejercicio 3

In [ ]:
def bigramas(frase):
    tokens = re.findall(r"\w+", frase.lower())
    return list(zip(tokens, tokens[1:]))

lista = bigramas(FRASE_NEGACION["spa"])
n_total = len(lista)
n_distintos = len(set(lista))
n_rasgos_1_2 = len(set(re.findall(r"\w+", FRASE_NEGACION["spa"].lower()))) + n_distintos

# Comprobación automática
assert n_total is not None and n_distintos is not None and n_rasgos_1_2 is not None and bigramas("a b") is not None, "Completa las líneas marcadas con TODO"
assert bigramas("uno dos tres") == [("uno", "dos"), ("dos", "tres")], "un bigrama es una pareja de palabras consecutivas"
assert n_total == 5 and n_distintos == 4, "la frase tiene 5 bigramas y el primero se repite"
assert n_rasgos_1_2 == 8, "4 palabras distintas + 4 bigramas distintos"
for idioma, frase in FRASE_NEGACION.items():
    cv_bi = CountVectorizer(token_pattern=r"\w+", ngram_range=(2, 2)).fit([frase])
    assert len(cv_bi.vocabulary_) == len(set(bigramas(frase))) == 4
    cv_12 = CountVectorizer(token_pattern=r"\w+", ngram_range=(1, 2)).fit([frase])
    assert len(cv_12.vocabulary_) == 8
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
u = np.array([2, 1, 0, 1])
v = np.array([1, 1, 1, 0])
w = 2 * u

def coseno(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

cos_uv = coseno(u, v)
cos_uw = coseno(u, w)
cos_vw = coseno(v, w)

# Comprobación automática
assert cos_uv is not None and cos_uw is not None and cos_vw is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(cos_uv, 3 / np.sqrt(18)), "u·v = 3, ‖u‖ = √6, ‖v‖ = √3"
assert np.isclose(cos_uw, 1.0), "w tiene la misma dirección que u"
assert np.isclose(cos_vw, cos_uv), "v se compara con la misma dirección, así que cos(v, w) = cos(v, u)"
esperada = cosine_similarity(np.array([u, v, w]))
assert np.allclose([cos_uv, cos_uw, cos_vw], [esperada[0, 1], esperada[0, 2], esperada[1, 2]]), "debe coincidir con cosine_similarity de scikit-learn"
assert cos_uw > cos_uv, "u está más cerca de w que de v"
print(f"Ejercicio 4 correcto ✔ (cos(u, v) = {cos_uv:.4f})")

### Solución del ejercicio 5

In [ ]:
def df_por_palabra(frases):
    return Counter(p for f in frases for p in set(re.findall(r"\w+", f.lower())))

def predecir(frases):
    n = len(frases)
    df = df_por_palabra(frases)
    base = len(df)
    min2 = sum(1 for d in df.values() if d >= 2)
    max075 = sum(1 for d in df.values() if d <= 0.75 * n)
    ambos = sum(1 for d in df.values() if 2 <= d <= 0.75 * n)
    return base, min2, max075, ambos

previsto = {idioma: predecir(FRASES[idioma]) for idioma in ("spa", "ita")}

# Comprobación automática
assert df_por_palabra(["a b", "a c"]) is not None and all(v is not None for t in previsto.values() for v in t), "Completa las líneas marcadas con TODO"
assert previsto["spa"] == (11, 4, 10, 3), "español: 11 palabras; con min_df=2 quedan la, fue, entrega y amable; con max_df=0.75 se va fue (está en las 4 frases)"
assert previsto["ita"] == (12, 8, 11, 7), "italiano: 12 palabras; con max_df=0.75 se va è (está en las 4 frases)"
for idioma in ("spa", "ita"):
    real = tuple(len(CountVectorizer(token_pattern=r"\w+", **p).fit(FRASES[idioma]).vocabulary_)
                 for p in ({}, {"min_df": 2}, {"max_df": 0.75}, {"min_df": 2, "max_df": 0.75}))
    assert real == previsto[idioma], f"{idioma}: tu previsión {previsto[idioma]} no coincide con CountVectorizer {real}"
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
IDIOMA = "spa"
df_res = pd.read_csv(cargar_datos(ARCHIVO_RESENAS[IDIOMA]))

vectorizador = TfidfVectorizer(token_pattern=r"\w+")
X = vectorizador.fit_transform(df_res["texto_limpio"])

def mas_similares(i, k=3):
    sim = cosine_similarity(X[i], X).ravel()
    sim[i] = -1
    return list(np.argsort(-sim)[:k])

def buscar(consulta, k=3):
    sim = cosine_similarity(vectorizador.transform([consulta]), X).ravel()
    return list(np.argsort(-sim)[:k])

assert vectorizador is not None and X is not None, "Completa las líneas marcadas con TODO"
assert sp.issparse(X) and X.shape[0] == len(df_res), "X debe ser una matriz dispersa con una fila por reseña"
assert X.nnz / (X.shape[0] * X.shape[1]) < 0.05, "la matriz debería tener menos del 5 % de celdas no nulas"
assert np.allclose(np.sqrt(X.multiply(X).sum(axis=1)).A1, 1.0), "las filas de TfidfVectorizer tienen norma L2 igual a 1"
for i in range(0, len(df_res), 25):
    vecinos = mas_similares(i, k=3)
    assert vecinos is not None and len(vecinos) == 3 and i not in list(vecinos), "mas_similares debe devolver 3 índices sin incluir la propia reseña"
    sim = cosine_similarity(X[i], X[list(vecinos)]).ravel()
    assert (np.diff(sim) <= 1e-12).all(), "los vecinos deben ir de mayor a menor similitud"
    assert buscar(df_res.loc[i, "texto_limpio"], k=1)[0] == i, "una reseña debe ser su propia vecina más parecida al buscarla"
consulta_nueva = df_res.loc[3, "texto_limpio"].lower()
assert buscar(consulta_nueva, k=1)[0] == 3, "el texto en minúsculas sigue encontrando la reseña original"
print("Reto final correcto ✔")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto de la clase no te ha quedado claro (bolsa de palabras, n-gramas, TF-IDF de scikit-learn frente al del libro, dispersión, coseno, `fit` frente a `transform`)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*